# NB4: Target Model — Llama-3-8B Loading & Inference

**Purpose:** Load Llama-3-8B-Instruct in 4-bit quantization, provide inference functions, and support loading LoRA-hardened model variants.

**Memory Budget:**
| Component | VRAM |
|---|---|
| Llama-3-8B (4-bit NF4) | ~5.5 GB |
| KV Cache (512 tokens) | ~0.5 GB |
| Inference overhead | ~1 GB |
| **Total inference** | **~7 GB** |

**Prerequisites:** Run NB1 first. API keys must be configured.

---

In [ ]:
# ============================================================
# Cell 0: Initialize
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/MediCS/utils')
from medics_utils import *

config = init_medics(mount_drive=False)
keys = load_api_keys()
openai_client = setup_api_clients(keys)
print("Ready.")

## Load Base Model (4-bit Quantized)

In [ ]:
# ============================================================
# Cell 1: Load Llama-3-8B-Instruct with 4-bit quantization
# ============================================================
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub import login

login(token=keys["hf_token"])

MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"

# Detect GPU type for optimal settings
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"
is_ampere_or_newer = torch.cuda.get_device_properties(0).major >= 8 if torch.cuda.is_available() else False

compute_dtype = torch.bfloat16 if is_ampere_or_newer else torch.float16
print(f"GPU: {gpu_name}")
print(f"Compute dtype: {compute_dtype}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

print(f"\nLoading {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=keys["hf_token"])
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # For batch generation

# Build model kwargs
model_kwargs = dict(
    quantization_config=bnb_config,
    device_map="auto",
    token=keys["hf_token"],
    torch_dtype=compute_dtype,
)

# Enable Flash Attention 2 on Ampere+ GPUs
if is_ampere_or_newer:
    try:
        model_kwargs["attn_implementation"] = "flash_attention_2"
        print("Flash Attention 2: enabled")
    except Exception:
        print("Flash Attention 2: not available, using default")

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs)
model.eval()

print(f"\nModel loaded successfully.")
print_gpu_memory()

## Inference Functions

In [ ]:
# ============================================================
# Cell 2: Define inference functions
# ============================================================

def generate_response(prompt, max_new_tokens=512, temperature=0.7,
                      conversation_history=None, _model=None, _tokenizer=None):
    """
    Generate a response from the target model.

    Args:
        prompt: The user message to respond to.
        max_new_tokens: Maximum tokens to generate.
        temperature: Sampling temperature.
        conversation_history: Optional list of {role, content} dicts for multi-turn.
        _model: Override model (for hardened model testing).
        _tokenizer: Override tokenizer.

    Returns:
        str: The model's response text.
    """
    _model = _model or model
    _tokenizer = _tokenizer or tokenizer

    if conversation_history is None:
        messages = [{"role": "user", "content": prompt}]
    else:
        messages = conversation_history + [{"role": "user", "content": prompt}]

    input_text = _tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = _tokenizer(input_text, return_tensors="pt").to(_model.device)

    with torch.no_grad():
        outputs = _model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
            pad_token_id=_tokenizer.eos_token_id,
        )

    response = _tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )
    return response.strip()


def generate_multiturn_response(turns, _model=None, _tokenizer=None):
    """
    Execute a multi-turn conversation. Returns the final response.

    Args:
        turns: List of user message strings.

    Returns:
        str: The final assistant response.
    """
    history = []
    final_response = ""

    for turn_prompt in turns:
        response = generate_response(
            turn_prompt,
            conversation_history=history,
            _model=_model,
            _tokenizer=_tokenizer,
        )
        history.append({"role": "user", "content": turn_prompt})
        history.append({"role": "assistant", "content": response})
        final_response = response

    return final_response


print("Inference functions defined.")

In [ ]:
# ============================================================
# Cell 3: Quick inference test
# ============================================================
test_prompts = [
    "What is the recommended daily dosage of ibuprofen for adults?",
    "How does acetaminophen work to reduce fever?",
    "What are the common side effects of amoxicillin?",
]

print("Testing inference on benign medical questions:\n")
for prompt in test_prompts:
    response = generate_response(prompt, max_new_tokens=150)
    print(f"Q: {prompt}")
    print(f"A: {response[:300]}...")
    print("-" * 60)

print("\nInference test complete.")
print_gpu_memory()

## Load LoRA-Hardened Model (for Defense Rounds 1+)

In [ ]:
# ============================================================
# Cell 4: Load a LoRA-hardened model variant
# Uncomment and set DEFENSE_ROUND to load a specific defense round.
# ============================================================
from peft import PeftModel

def load_hardened_model(base_model, defense_round: int):
    """
    Load a LoRA-fine-tuned (hardened) version of the model.

    Args:
        base_model: The base Llama-3 model.
        defense_round: Which defense round's adapter to load.

    Returns:
        The model with LoRA adapter applied.
    """
    adapter_path = f"{PATHS['checkpoints']}/lora_round_{defense_round}"

    if not os.path.exists(adapter_path):
        print(f"ERROR: No adapter found at {adapter_path}")
        print(f"Run NB5 (Defense) for round {defense_round} first.")
        return None

    print(f"Loading LoRA adapter from {adapter_path}...")
    hardened = PeftModel.from_pretrained(base_model, adapter_path)
    print(f"Adapter loaded. Merging weights...")
    hardened = hardened.merge_and_unload()
    hardened.eval()
    print(f"Hardened model (round {defense_round}) ready.")
    print_gpu_memory()
    return hardened


# --- Uncomment below to load a hardened model ---
# DEFENSE_ROUND = 0
# hardened_model = load_hardened_model(model, DEFENSE_ROUND)
# if hardened_model:
#     # Replace the global model for NB3 to use
#     model = hardened_model
#     print(f"Global model replaced with defense round {DEFENSE_ROUND}.")

## Batch Inference Utility

In [ ]:
# ============================================================
# Cell 5: Batch inference for evaluation
# ============================================================

def batch_inference(prompts, output_path=None, max_new_tokens=512,
                    temperature=0.7, _model=None, _tokenizer=None):
    """
    Run inference on a list of prompts with optional JSONL logging.

    Args:
        prompts: List of dicts with at least {"id": str, "prompt": str}
        output_path: Optional JSONL path for saving results.

    Returns:
        List of dicts with added "response" field.
    """
    # Resume from checkpoint
    completed_ids = set()
    if output_path and os.path.exists(output_path):
        existing = read_jsonl(output_path)
        completed_ids = {r["id"] for r in existing}

    remaining = [p for p in prompts if p["id"] not in completed_ids]
    print(f"Batch inference: {len(completed_ids)} done, {len(remaining)} remaining")

    results = []
    for i, item in enumerate(tqdm(remaining)):
        try:
            response = generate_response(
                item["prompt"],
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                _model=_model,
                _tokenizer=_tokenizer,
            )
            result = {**item, "response": response,
                      "timestamp": datetime.now().isoformat()}
            results.append(result)

            if output_path:
                append_jsonl(output_path, result)

        except Exception as e:
            print(f"  Error on {item['id']}: {e}")
            # Clear GPU cache on OOM
            if "out of memory" in str(e).lower():
                torch.cuda.empty_cache()

    return results


print("Batch inference utility defined.")
print("\n" + "=" * 60)
print("NB4 COMPLETE — Model loaded and inference functions ready.")
print("Proceed to NB3 (Red Team) or NB5 (Defense).")
print("=" * 60)